# PubMedQA Classification Pipeline
**Embeddings**: `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext`  
**Classifiers**: SVM · Decision Tree · Random Forest  
**Task**: 3-class — `yes / no / maybe`  
**Data**: PQA-L (1,000 expert-labeled samples only)

---
## 1. Install dependencies

In [1]:
!pip install -q transformers datasets scikit-learn torch

---
## 2. Imports

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch : {torch.__version__}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {DEVICE}")

PyTorch : 2.10.0+cu128
Device  : cuda


---
## 3. Configuration

In [3]:
MODEL_NAME  = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
MAX_LENGTH  = 512
BATCH_SIZE  = 16    # reduce to 8 if you hit OOM
TEST_SIZE   = 0.2
RANDOM_SEED = 42

LABEL_MAP = {"yes": 0, "no": 1, "maybe": 2}
LABEL_NAMES = ["yes", "no", "maybe"]

---
## 4. Load PQA-L (labeled dataset)

In [4]:
print("Loading PubMedQA labeled split (pqa_labeled)...")
dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")
print(f"Total samples : {len(dataset)}")

from collections import Counter
dist = Counter(s["final_decision"] for s in dataset)
print(f"Label distribution: {dict(dist)}")

# Inspect one sample
s = dataset[0]
print(f"\nSample fields : {list(s.keys())}")
print(f"Question      : {s['question'][:100]}")
print(f"Label         : {s['final_decision']}")

Loading PubMedQA labeled split (pqa_labeled)...


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Total samples : 1000
Label distribution: {'yes': 552, 'no': 338, 'maybe': 110}

Sample fields : ['pubid', 'question', 'context', 'long_answer', 'final_decision']
Question      : Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Label         : yes


---
## 5. Preprocess — build input texts and labels

In [5]:
def build_text(sample: dict) -> str:
    """
    Concatenate question + all context passages.
    Format: question [SEP] passage1 passage2 ...
    """
    question = sample["question"].strip()
    passages = " ".join(sample["context"]["contexts"])
    return f"{question} [SEP] {passages}"


texts  = [build_text(s) for s in dataset]
labels = [LABEL_MAP[s["final_decision"].lower()] for s in dataset]

print(f"Built {len(texts)} input texts.")
print(f"Sample text (first 200 chars): {texts[0][:200]}...")

Built 1000 input texts.
Sample text (first 200 chars): Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? [SEP] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponoge...


---
## 6. Extract PubMedBERT embeddings

We use the **`[CLS]` token** from the last hidden state as a fixed-size (768-dim) representation of each sample.

In [6]:
print(f"Loading tokenizer and model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()
print(f"Model loaded. Hidden size: {bert_model.config.hidden_size}")


def extract_embeddings(texts: list, batch_size: int = BATCH_SIZE) -> np.ndarray:
    """
    Tokenize texts in batches and extract the [CLS] embedding from
    the last hidden state of PubMedBERT.

    Returns: np.ndarray of shape (N, 768)
    """
    all_embeddings = []
    n_batches = (len(texts) + batch_size - 1) // batch_size

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            batch_num   = i // batch_size + 1

            encoded = tokenizer(
                batch_texts,
                max_length=MAX_LENGTH,
                padding=True,
                truncation=True,
                return_tensors="pt",
            )
            input_ids      = encoded["input_ids"].to(DEVICE)
            attention_mask = encoded["attention_mask"].to(DEVICE)

            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            # [CLS] token = first token of last hidden state
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embeddings)

            if batch_num % 5 == 0 or batch_num == n_batches:
                print(f"  Batch {batch_num}/{n_batches}")

    return np.vstack(all_embeddings)


print("\nExtracting embeddings (this may take a few minutes on CPU)...")
X = extract_embeddings(texts)
y = np.array(labels)

print(f"\nEmbedding matrix shape : {X.shape}")
print(f"Label array shape      : {y.shape}")
print(f"Label distribution     : { {LABEL_NAMES[k]: int((y==k).sum()) for k in range(3)} }")

Loading tokenizer and model: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Hidden size: 768

Extracting embeddings (this may take a few minutes on CPU)...
  Batch 5/63
  Batch 10/63
  Batch 15/63
  Batch 20/63
  Batch 25/63
  Batch 30/63
  Batch 35/63
  Batch 40/63
  Batch 45/63
  Batch 50/63
  Batch 55/63
  Batch 60/63
  Batch 63/63

Embedding matrix shape : (1000, 768)
Label array shape      : (1000,)
Label distribution     : {'yes': 552, 'no': 338, 'maybe': 110}


---
## 7. Train / test split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y,
)

print(f"Train samples : {len(X_train)}")
print(f"Test  samples : {len(X_test)}")

Train samples : 800
Test  samples : 200


---
## 8. Define classifiers

In [8]:
classifiers = {
    "SVM (RBF kernel)": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        random_state=RANDOM_SEED,
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=None,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_SEED,
    ),
}

print("Classifiers defined:")
for name in classifiers:
    print(f"  - {name}")

Classifiers defined:
  - SVM (RBF kernel)
  - Decision Tree
  - Random Forest


---
## 9. Train and evaluate all classifiers

In [9]:
results = {}

for name, clf in classifiers.items():
    print(f"\n{'='*60}")
    print(f"Classifier: {name}")
    print(f"{'='*60}")

    # Train
    print(f"Training...")
    clf.fit(X_train, y_train)

    # Predict
    y_pred_train = clf.predict(X_train)
    y_pred_test  = clf.predict(X_test)

    # Metrics
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc  = accuracy_score(y_test,  y_pred_test)
    macro_f1  = f1_score(y_test, y_pred_test, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_test, y_pred_test, average="weighted", zero_division=0)

    results[name] = {
        "train_acc"   : train_acc,
        "test_acc"    : test_acc,
        "macro_f1"    : macro_f1,
        "weighted_f1" : weighted_f1,
        "y_pred"      : y_pred_test,
    }

    print(f"Train accuracy : {train_acc:.4f}")
    print(f"Test  accuracy : {test_acc:.4f}")
    print(f"Macro F1       : {macro_f1:.4f}")
    print(f"Weighted F1    : {weighted_f1:.4f}")
    print()
    print(classification_report(
        y_test, y_pred_test,
        target_names=LABEL_NAMES, digits=4
    ))
    print("Confusion Matrix (rows=true, cols=pred):")
    cm = confusion_matrix(y_test, y_pred_test, labels=[0, 1, 2])
    header = "          " + "  ".join(f"{l:>7}" for l in LABEL_NAMES)
    print(header)
    for i, row in enumerate(cm):
        print(f"  {LABEL_NAMES[i]:>5}  " + "  ".join(f"{v:7d}" for v in row))


Classifier: SVM (RBF kernel)
Training...
Train accuracy : 0.5663
Test  accuracy : 0.4500
Macro F1       : 0.3814
Weighted F1    : 0.4593

              precision    recall  f1-score   support

         yes     0.6667    0.3818    0.4855       110
          no     0.4356    0.6471    0.5207        68
       maybe     0.1111    0.1818    0.1379        22

    accuracy                         0.4500       200
   macro avg     0.4045    0.4036    0.3814       200
weighted avg     0.5270    0.4500    0.4593       200

Confusion Matrix (rows=true, cols=pred):
              yes       no    maybe
    yes       42       48       20
     no       12       44       12
  maybe        9        9        4

Classifier: Decision Tree
Training...
Train accuracy : 1.0000
Test  accuracy : 0.4550
Macro F1       : 0.3406
Weighted F1    : 0.4511

              precision    recall  f1-score   support

         yes     0.5826    0.6091    0.5956       110
          no     0.3438    0.3235    0.3333        68

---
## 10. Comparison summary

In [10]:
print(f"\n{'Classifier':<25}  {'Train Acc':>10}  {'Test Acc':>9}  {'Macro F1':>9}  {'Weighted F1':>12}")
print("-" * 72)
for name, r in results.items():
    print(
        f"{name:<25}  "
        f"{r['train_acc']:>10.4f}  "
        f"{r['test_acc']:>9.4f}  "
        f"{r['macro_f1']:>9.4f}  "
        f"{r['weighted_f1']:>12.4f}"
    )

best = max(results.items(), key=lambda x: x[1]["macro_f1"])
print(f"\nBest classifier by Macro F1: {best[0]} ({best[1]['macro_f1']:.4f})")


Classifier                  Train Acc   Test Acc   Macro F1   Weighted F1
------------------------------------------------------------------------
SVM (RBF kernel)               0.5663     0.4500     0.3814        0.4593
Decision Tree                  1.0000     0.4550     0.3406        0.4511
Random Forest                  1.0000     0.5600     0.2653        0.4202

Best classifier by Macro F1: SVM (RBF kernel) (0.3814)


---
## 11. Inference on a new question

In [11]:
def predict(question: str, context_passages: list, classifier_name: str = "SVM (RBF kernel)"):
    """
    Predict yes/no/maybe for a new biomedical question.

    Args:
        question          : The research question.
        context_passages  : List of relevant abstract sentences.
        classifier_name   : One of the trained classifiers.

    Returns:
        str: 'yes', 'no', or 'maybe'
    """
    text = question.strip() + " [SEP] " + " ".join(context_passages)
    emb  = extract_embeddings([text], batch_size=1)      # shape: (1, 768)
    clf  = classifiers[classifier_name]
    pred = clf.predict(emb)[0]
    return LABEL_NAMES[pred]


# --- Demo ---
demo_question = "Does aerobic exercise improve cognitive function in older adults?"
demo_context  = [
    "Aerobic exercise has been shown to increase hippocampal volume.",
    "Several randomized trials found significant improvements in executive function.",
    "Meta-analyses confirm a positive effect on memory and processing speed.",
]

for clf_name in classifiers:
    answer = predict(demo_question, demo_context, clf_name)
    print(f"{clf_name:<25}  →  {answer}")

  Batch 1/1
SVM (RBF kernel)           →  yes
  Batch 1/1
Decision Tree              →  yes
  Batch 1/1
Random Forest              →  yes
